In [ ]:
# Veriyi Anlamak ve Hedef Değişkeni (RUL) Oluşturmak
# İndirdiğimiz NASA CMAPSS veri setinde motorların her "cycle" (uçuş döngüsü)
# için sensör değerleri bulunur. Ancak veri setinde doğrudan "Kalan Ömür"
# sütunu yoktur. Bunu bizim hesaplamamız gerekir.
# Neden? Çünkü makine öğrenmesi modeline "bak bu satırdaki sensör değerleri varken
# motorun bitmesine 50 uçuş kalmıştı" dememiz lazım ki model bu örüntüyü öğrensin.

# Yapılacak İşlemler:

#     Veriyi pandas ile oku.
#     Her motorun (unit_number) maksimum uçuş döngüsünü bul.
#     Mevcut döngüyü maksimumdan çıkararak RUL sütununu oluştur.

In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
index_names = ["unit_number", "time_cycle"]
setting_names = ["setting_1", "setting_2", "setting_3"]
sensor_names = ["s_{}".format(i) for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

In [4]:
data_path = "~/Coding/SkyGuard AI/CMAPSSData/train_FD001.txt"

In [5]:
train_df = pd.read_csv(data_path, sep="\s+", header=None, names=col_names)

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_22520/3819127536.py:1: SyntaxWarning: invalid escape sequence '\s'
  train_df = pd.read_csv(data_path, sep="\s+", header=None, names=col_names)


In [6]:
def add_remaining_useful_life(df):
    # Her bir motor (unit_number) için maksimum uçuş döngüsünü bul
    max_cycle = df.groupby("unit_number")["time_cycle"].transform("max")

    # RUL = Maksimum Döngü - Mevcut Döngü
    df["RUL"] = max_cycle - df["time_cycle"]
    return df

In [7]:
train_df = add_remaining_useful_life(train_df)

# İlk 5 satıra bakalım: Her satırda RUL'un azalmalı
print(train_df[["unit_number", "time_cycle", "RUL"]].head())

   unit_number  time_cycle  RUL
0            1           1  191
1            1           2  190
2            1           3  189
3            1           4  188
4            1           5  187


In [8]:
train_df.head(100)

,unit_number,time_cycle,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1,96,-0.0034,0.0001,100.0,518.67,642.19,1584.07,1395.16,14.62,...,2388.06,8130.69,8.4311,0.03,392,2388,100.0,38.88,23.3255,96
96,1,97,0.0035,-0.0003,100.0,518.67,642.07,1595.77,1407.81,14.62,...,2388.06,8128.74,8.4105,0.03,392,2388,100.0,39.01,23.2963,95
97,1,98,0.0006,0.0004,100.0,518.67,642.00,1591.11,1404.56,14.62,...,2388.06,8127.89,8.4012,0.03,391,2388,100.0,38.96,23.2554,94
98,1,99,-0.0005,-0.0000,100.0,518.67,642.46,1592.73,1406.13,14.62,...,2388.10,8131.77,8.4481,0.03,393,2388,100.0,38.82,23.2323,93


In [9]:
drop_sensors = ["s_1", "s_5", "s_6", "s_10", "s_16", "s_18", "s_19"]
train_df.drop(labels=drop_sensors, axis=1, inplace=True)

### save processed data as a new file

In [10]:
# os.makedirs("processed_data/", exist_ok=True)

In [11]:
train_df.to_csv(
    "~/Coding/SkyGuard AI/CMAPSSData/processed_data/train_FD001_preprocessed.csv",
    index=False,
)